In [49]:
import os
import yaml
import numpy as np
import polars as pl
import pandas as pd
from tqdm import tqdm
import statsmodels.api as sm
import pyranges as pr

from plotnine import *
import matplotlib.pyplot as plt
plt.rcParams['svg.fonttype'] = 'none'

In [ ]:
BENCH_DIR = "/home/dnanexus/data_dir/other_benchmarks"
# BENCH_DIR = "/s/project/deeprvat/ukb_gym/other_benchmarks"

!mkdir -p {BENCH_DIR}

: 

# UKBBGym

In [ ]:
RAP_ANNO_DIR = "project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/wgs/qced_maf1e-3_loftee_olink_genes/qced_maf1e-3_loftee_olink_genes_EURunrelated"
LOCAL_ANNO_DIR = "/home/dnanexus/data_dir"

# ANNO_FILE = "annotations_fillna_ukbgym_with_mane.parquet"
ANNO_FILE = "annotations_with_all.parquet"

!dx download {RAP_ANNO_DIR}/{ANNO_FILE} -o {LOCAL_ANNO_DIR}/{ANNO_FILE}
anno = pl.scan_parquet(f"{LOCAL_ANNO_DIR}/{ANNO_FILE}")

mac = 20

consequence_columns = [c for c in anno.collect_schema().names() if c.startswith("consequence_")]
encode_columns = [c for c in anno.collect_schema().names() if c.startswith("encode_")]

anno = (
    anno
    .with_columns(
        aa_pos = pl.col('protein_position').str.split("/").list.get(0),
        ref_aa = pl.col('amino_acids').str.split("/").list.get(0, null_on_oob=True),
        alt_aa = pl.col('amino_acids').str.split("/").list.get(1, null_on_oob=True),
    )
    .select([
        'chrom', 
        'pos', 
        'ref', 
        'alt',
        'id', 
        'region',
        'aa_pos',
        'ref_aa',
        'alt_aa',
        'loftee_hc',
        'loftee_lc',
    ]
    + consequence_columns
    + encode_columns
    )
    .unique()
    .collect(engine='streaming')
)

anno

Error: path "/home/dnanexus/data_dir/annotations_with_all.parquet" already
exists but -f/--overwrite was not set


In [ ]:
anno.filter(pl.col('amino_acids').is_not_null())

ColumnNotFoundError: unable to find column "amino_acids"; valid columns: ["chrom", "pos", "ref", "alt", "id", "region", "aa_pos", "ref_aa", "alt_aa", "loftee_hc", "loftee_lc", "consequence_3_prime_utr_variant", "consequence_5_prime_utr_variant", "consequence_coding_sequence_variant", "consequence_downstream_gene_variant", "consequence_frameshift_variant", "consequence_incomplete_terminal_codon_variant", "consequence_inframe_deletion", "consequence_inframe_insertion", "consequence_intron_variant", "consequence_missense_variant", "consequence_protein_altering_variant", "consequence_splice_acceptor_variant", "consequence_splice_donor_5th_base_variant", "consequence_splice_donor_region_variant", "consequence_splice_donor_variant", "consequence_splice_polypyrimidine_tract_variant", "consequence_splice_region_variant", "consequence_start_lost", "consequence_start_retained_variant", "consequence_stop_gained", "consequence_stop_lost", "consequence_stop_retained_variant", "consequence_synonymou

# ClinVar

In [ ]:
# Download ClinVar VCF (GRCh38) and its index
!mkdir -p {BENCH_DIR}/clinvar
!wget -nc https://ftp.ncbi.nlm.nih.gov/pub/clinvar/vcf_GRCh38/clinvar.vcf.gz -O {BENCH_DIR}/clinvar/clinvar.vcf.gz
!wget -nc https://ftp.ncbi.nlm.nih.gov/pub/clinvar/vcf_GRCh38/clinvar.vcf.gz.tbi -O {BENCH_DIR}/clinvar/clinvar.vcf.gz.tbi

File ‘/home/dnanexus/data_dir/other_benchmarks/clinvar/clinvar.vcf.gz’ already there; not retrieving.
File ‘/home/dnanexus/data_dir/other_benchmarks/clinvar/clinvar.vcf.gz.tbi’ already there; not retrieving.


In [ ]:
# 1. Create a dummy dataframe with your example string (added CLNSIG for demonstration)
data = {
    "info_col": [
        ".11:g.917887G>T;CLNVC=single_nucleotide_variant;CLNVCSO=SO:0001483;GENEINFO=LINC02593:100130417;MC=SO:0001627|intron_variant;ORIGIN=0",
        "CLNVC=single_nucleotide_variant;CLNSIG=Likely_benign;MC=SO:0001583|missense_variant;ORIGIN=1",
        "CLNVC=deletion;CLNSIG=Pathogenic;MC=SO:0001587|stop_gained;ORIGIN=1"
    ]
}
df = pl.DataFrame(data)

# 2. Extract the fields using Regex
# We create two new columns by parsing the 'info_col'
df.with_columns(
    # Regex explanation for MC: 
    # Look for "MC=", skip characters until pipe "|", capture text until next comma or semicolon
    variant_region = pl.col("info_col").str.extract(r"MC=[^|]+\|([^;,]+)", 1),
    
    # Regex explanation for CLNSIG: 
    # Look for "CLNSIG=", capture everything until the next semicolon
    clinical_significance = pl.col("info_col").str.extract(r"CLNSIG=([^;]+)", 1)
)

info_col,variant_region,clinical_significance
str,str,str
""".11:g.917887G>T;CLNVC=single_n…","""intron_variant""",null
"""CLNVC=single_nucleotide_varian…","""missense_variant""","""Likely_benign"""
"""CLNVC=deletion;CLNSIG=Pathogen…","""stop_gained""","""Pathogenic"""


In [ ]:
clinvar = (
    pl.read_csv(f"{BENCH_DIR}/clinvar/clinvar.vcf.gz", comment_prefix="##", separator="\t", ignore_errors=True)
    .with_columns(
        chrom = 'chr' + pl.col("#CHROM").fill_null("NA").cast(str),
    
        # Look for "CLNSIG=", capture everything until the next semicolon
        clinical_significance = pl.col("INFO").str.extract(r"CLNSIG=([^;]+)", 1),
    )
    .drop(['#CHROM', 'ID', 'QUAL', 'FILTER'])
    .rename({
        "POS": "pos",
        "REF": "ref",
        "ALT": "alt",
        "INFO": "info",
    })
    .with_columns(
        id = pl.col("chrom") + ":" + pl.col("pos").cast(str) + ":" + pl.col("ref") + ":" + pl.col("alt"),

        # Look for "MC=", skip characters until pipe "|", capture text until next comma or semicolon
        variant_region = pl.col("info").str.extract(r"MC=[^|]+\|([^;,]+)", 1),
    )
    # .with_columns(
    #     one_hot = pl.lit(1)
    # )
    # .pivot(
    #     index=['id'],
    #     on='clinical_significance',
    #     values='one_hot',          # or whichever column you want to spread
    #     aggregate_function='first',
    # )
    # .fill_null(0)
)

clinvar

pos,ref,alt,info,chrom,clinical_significance,id,variant_region
i64,str,str,str,str,str,str,str
66926,"""AG""","""A""","""ALLELEID=3544463;CLNDISDB=Huma…","""chr1""","""Uncertain_significance""","""chr1:66926:AG:A""","""intron_variant"""
69134,"""A""","""G""","""ALLELEID=2193183;CLNDISDB=MedG…","""chr1""","""Likely_benign""","""chr1:69134:A:G""","""missense_variant"""
69241,"""C""","""T""","""ALLELEID=4679177;CLNDISDB=MedG…","""chr1""","""Uncertain_significance""","""chr1:69241:C:T""","""missense_variant"""
69308,"""A""","""G""","""ALLELEID=4039319;CLNDISDB=MedG…","""chr1""","""Uncertain_significance""","""chr1:69308:A:G""","""missense_variant"""
69314,"""T""","""G""","""ALLELEID=3374047;CLNDISDB=MedG…","""chr1""","""Uncertain_significance""","""chr1:69314:T:G""","""missense_variant"""
…,…,…,…,…,…,…,…
274185,"""C""","""T""","""ALLELEID=3894028;CLNDISDB=MedG…","""chrNA""","""Likely_benign""","""chrNA:274185:C:T""","""intron_variant"""
274366,"""G""","""C""","""ALLELEID=2200058;CLNDISDB=MedG…","""chrNA""","""Uncertain_significance""","""chrNA:274366:G:C""","""missense_variant"""
275068,"""T""","""C""","""ALLELEID=2226217;CLNDISDB=MedG…","""chrNA""","""Uncertain_significance""","""chrNA:275068:T:C""","""missense_variant"""


In [ ]:
anno = (
    anno
    .join(
        clinvar, 
        on="id", 
        how="left"
    )
)

anno

chrom,pos,ref,alt,id,region,aa_pos,loftee_hc,loftee_lc,consequence_3_prime_utr_variant,consequence_5_prime_utr_variant,consequence_coding_sequence_variant,consequence_downstream_gene_variant,consequence_frameshift_variant,consequence_incomplete_terminal_codon_variant,consequence_inframe_deletion,consequence_inframe_insertion,consequence_intron_variant,consequence_missense_variant,consequence_protein_altering_variant,consequence_splice_acceptor_variant,consequence_splice_donor_5th_base_variant,consequence_splice_donor_region_variant,consequence_splice_donor_variant,consequence_splice_polypyrimidine_tract_variant,consequence_splice_region_variant,consequence_start_lost,consequence_start_retained_variant,consequence_stop_gained,consequence_stop_lost,consequence_stop_retained_variant,consequence_synonymous_variant,consequence_upstream_gene_variant,encode_dels,encode_ca-ctcf,encode_ca,encode_ca-h3k4me3,encode_tf,encode_ca-tf,encode_pels,encode_pls,pos_right,ref_right,alt_right,info,chrom_right,clinical_significance,variant_region
str,i64,str,str,str,str,str,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,bool,bool,bool,bool,bool,bool,bool,bool,i64,str,str,str,str,str,str
"""chr1""",245042423,"""A""","""G""","""chr1:245042423:A:G""","""ENSG00000203666""",null,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,false,false,false,false,false,false,true,false,null,null,null,null,null,null,null
"""chr15""",100035084,"""A""","""AAG""","""chr15:100035084:A:AAG""","""ENSG00000140470""",null,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,false,false,false,false,false,false,false,false,null,null,null,null,null,null,null
"""chr1""",235113386,"""G""","""A""","""chr1:235113386:G:A""","""ENSG00000173726""",null,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,true,false,false,false,false,false,false,false,null,null,null,null,null,null,null
"""chr12""",8478736,"""C""","""T""","""chr12:8478736:C:T""","""ENSG00000205846""",null,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,false,false,false,false,false,false,false,false,null,null,null,null,null,null,null
"""chr5""",11802580,"""A""","""AG""","""chr5:11802580:A:AG""","""ENSG00000169862""",null,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,false,false,false,false,false,false,false,false,null,null,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""chr7""",92074650,"""G""","""A""","""chr7:92074650:G:A""","""ENSG00000127914""",null,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,false,false,false,false,false,false,false,false,null,null,null,null,null,null,null
"""chr11""",99946762,"""A""","""C""","""chr11:99946762:A:C""","""ENSG00000149972""",null,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,false,false,false,false,false,false,false,false,null,null,null,null,null,null,null
"""chr21""",28937254,"""C""","""G""","""chr21:28937254:C:G""","""ENSG00000198862""",null,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,false,false,false,false,false,false,false,false,null,null,null,null,null,null,null


## ProteinGym

In [ ]:
VERSION = "v1.3"
FILENAME = "DMS_ProteinGym_substitutions.zip"
TARGET_DIR = f"{BENCH_DIR}/protein_gym"
TARGET_FILE = f"{TARGET_DIR}/{FILENAME}"

# 1. Create Directory
!mkdir -p {TARGET_DIR}

# 2. Download only if missing
if not os.path.exists(TARGET_FILE):
    print("File not found. Downloading...")
    !curl -o {TARGET_FILE} https://marks.hms.harvard.edu/proteingym/ProteinGym_{VERSION}/{FILENAME}
else:
    print("File already exists. Skipping download.")

# 3. Unzip with "Yes to All" (-o)
# We use -o to overwrite if it exists, avoiding the interactive prompt
print("Unzipping...")
!unzip -o {TARGET_FILE} -d {TARGET_DIR}

File already exists. Skipping download.
Unzipping...
Archive:  /home/dnanexus/data_dir/other_benchmarks/protein_gym/DMS_ProteinGym_substitutions.zip
  inflating: /home/dnanexus/data_dir/other_benchmarks/protein_gym/DMS_ProteinGym_substitutions/SDA_BACSU_Tsuboyama_2023_1PV0.csv  
  inflating: /home/dnanexus/data_dir/other_benchmarks/protein_gym/DMS_ProteinGym_substitutions/PAI1_HUMAN_Huttinger_2021.csv  
  inflating: /home/dnanexus/data_dir/other_benchmarks/protein_gym/DMS_ProteinGym_substitutions/S22A1_HUMAN_Yee_2023_activity.csv  
  inflating: /home/dnanexus/data_dir/other_benchmarks/protein_gym/DMS_ProteinGym_substitutions/HIS7_YEAST_Pokusaeva_2019.csv  
  inflating: /home/dnanexus/data_dir/other_benchmarks/protein_gym/DMS_ProteinGym_substitutions/AMIE_PSEAE_Wrenbeck_2017.csv  
  inflating: /home/dnanexus/data_dir/other_benchmarks/protein_gym/DMS_ProteinGym_substitutions/ACE2_HUMAN_Chan_2020.csv  
  inflating: /home/dnanexus/data_dir/other_benchmarks/protein_gym/DMS_ProteinGym_substi

In [ ]:
csv_pattern = f"{TARGET_DIR}/DMS_ProteinGym_substitutions/*HUMAN*.csv"

print(f"Scanning files matching: {csv_pattern}")

proteingym_snp = (
    pl.scan_csv(csv_pattern, include_file_paths="source_file")
    .with_columns(
        assay_id = pl.col("source_file")
                    .str.split("/")
                    .list.last()
                    .str.strip_suffix(".csv")
    )
    .with_columns(
        pl.col('mutant').str.extract_groups(r'([A-Z])(\d+)([A-Z])').struct.rename_fields(['aa_ref', 'aa_pos', 'aa_alt'])
    ).unnest('mutant')
    .with_columns(
        gene_symbol = pl.col("assay_id").str.split("_").list.get(0),
    )
    .select(['aa_ref', 'aa_pos', 'aa_alt', 'gene_symbol', 'DMS_score'])
    .unique()
    .collect(engine='streaming')
)

proteingym_snp

Scanning files matching: /home/dnanexus/data_dir/other_benchmarks/protein_gym/DMS_ProteinGym_substitutions/*HUMAN*.csv


aa_ref,aa_pos,aa_alt,gene_symbol,DMS_score
str,str,str,str,f64
"""Y""","""681""","""F""","""A4""",-0.868462
"""G""","""35""","""T""","""ADRB2""",2.564882
"""F""","""101""","""W""","""ADRB2""",2.178934
"""W""","""286""","""L""","""ADRB2""",1.224486
"""D""","""28""","""G""","""AMFR""",-2.3182
…,…,…,…,…
"""E""","""63""","""D""","""HECD1""",-2.259223
"""I""","""91""","""Y""","""HMDH""",0.8611
"""V""","""91""","""Q""","""HXK4""",0.745898


In [ ]:
# Gene name mapping from gnomAD constraint metrics (HGNC symbol -> ENSG ID)
gene_mapping = (
    pl.read_csv(
        '/home/dnanexus/data_dir/gnomad.v4.1.constraint_metrics.tsv',
        separator='\t', null_values='NA', columns=['gene', 'gene_id']
    )
    .filter(pl.col('gene_id').str.starts_with('ENSG'))
    .rename({'gene': 'gene_symbol', 'gene_id': 'region'})
    .unique()
)

# Manual mapping for UniProt mnemonic names -> HGNC symbols
# ProteinGym filenames use UniProt gene names which often differ from HGNC
uniprot_to_hgnc = {
    'A4': 'APP', 'B2L11': 'BCL2L11', 'CAR11': 'CARD11', 'CBPA2': 'CPA2',
    'CP2C9': 'CYP2C9', 'DNJA1': 'DNAJA1', 'GLPA': 'GYPA',
    'HECD1': 'HECTD1', 'HEM3': 'HMBS', 'HMDH': 'HMGCR', 'HXK4': 'GCK',
    'LYAM1': 'SELL', 'MK01': 'MAPK1', 'MTHR': 'MTHFR', 'NKX31': 'NKX3-1',
    'NUD15': 'NUDT15', 'OPSD': 'RHO', 'OTU7A': 'OTUD7A', 'P53': 'TP53',
    'PAI1': 'SERPINE1', 'PR40A': 'PRPF40A', 'RASH': 'HRAS', 'RASK': 'KRAS',
    'RD23A': 'RAD23A', 'S22A1': 'SLC22A1', 'SC6A4': 'SLC6A4', 'SERC': 'PSAT1',
    'SRBS1': 'SORBS1', 'SYUA': 'SNCA', 'TADBP': 'TARDBP', 'TPOR': 'MPL',
    'UBC9': 'UBE2I', 'VKOR1': 'VKORC1',
}
# Unmapped (chrX / not in gnomAD): ACE2, CBS, OTC, GDIA->GDI1, Q53Z42 (HLA)

# Remap ProteinGym gene symbols: first try HGNC lookup, then use as-is
proteingym_remapped = proteingym_snp.with_columns(
    hgnc_symbol = pl.col('gene_symbol').replace_strict(uniprot_to_hgnc, default=pl.col('gene_symbol'))
)

# Map to ENSG IDs via gnomAD
proteingym_mapped = (
    proteingym_remapped
    .join(
        gene_mapping.rename({'gene_symbol': 'hgnc_symbol'}),
        on='hgnc_symbol',
        how='inner',
    )
)

proteingym_mapped

aa_ref,aa_pos,aa_alt,gene_symbol,DMS_score,hgnc_symbol,region
str,str,str,str,f64,str,str
"""Y""","""681""","""F""","""A4""",-0.868462,"""APP""","""ENSG00000142192"""
"""G""","""35""","""T""","""ADRB2""",2.564882,"""ADRB2""","""ENSG00000169252"""
"""F""","""101""","""W""","""ADRB2""",2.178934,"""ADRB2""","""ENSG00000169252"""
"""W""","""286""","""L""","""ADRB2""",1.224486,"""ADRB2""","""ENSG00000169252"""
"""D""","""28""","""G""","""AMFR""",-2.3182,"""AMFR""","""ENSG00000159461"""
…,…,…,…,…,…,…
"""E""","""63""","""D""","""HECD1""",-2.259223,"""HECTD1""","""ENSG00000092148"""
"""I""","""91""","""Y""","""HMDH""",0.8611,"""HMGCR""","""ENSG00000113161"""
"""V""","""91""","""Q""","""HXK4""",0.745898,"""GCK""","""ENSG00000106633"""


In [48]:
# Create amino acid mutation keys from UKBgym annotations for matching
# amino_acids format: "A/C", protein_position format: "673/770"
anno_mutant_keys = (
    pl.scan_parquet(f"{LOCAL_ANNO_DIR}/{ANNO_FILE}")
    .select(['id', 'region', 'amino_acids', 'protein_position'])
    .filter(
        pl.col('amino_acids').is_not_null(),
        pl.col('protein_position').is_not_null(),
    )
    .with_columns(
        aa_pos = pl.col('protein_position').str.split('/').list.get(0),
        ref_aa = pl.col('amino_acids').str.split('/').list.get(0),
        alt_aa = pl.col('amino_acids').str.split('/').list.get(1),
    )
    .filter(
        pl.col('ref_aa').str.len_chars() == 1,
        pl.col('alt_aa').str.len_chars() == 1,
    )
    .with_columns(
        mutant = pl.col('ref_aa') + pl.col('aa_pos') + pl.col('alt_aa')
    )
    .select(['id', 'region', 'mutant'])
    .collect(engine='streaming')
)

# Merge ProteinGym with UKBgym via exact mutant + region match
# Average DMS_score per mutant+region in case of multiple assays
proteingym_to_ukbgym = (
    anno_mutant_keys
    .join(
        proteingym_mapped.select(['mutant', 'region', 'DMS_score', 'DMS_score_bin', 'assay_id']),
        on=['mutant', 'region'],
        how='inner'
    )
)

print(f"Matched {proteingym_to_ukbgym['id'].n_unique()} UKBgym variants to ProteinGym")

# Add ProteinGym DMS scores to main annotation dataframe
anno = (
    anno
    .join(
        proteingym_to_ukbgym
        .group_by('id')
        .agg(
            DMS_score = pl.col('DMS_score').mean(),
            DMS_score_bin = pl.col('DMS_score_bin').first(),
        ),
        on='id',
        how='left',
    )
)

anno

ComputeError: get index is out of bounds

# Plotting

In [ ]:
# Load gene-trait associations (670 significant gene-phenotype pairs)
gene_trait_df = (
    pl.read_parquet(f'{LOCAL_ANNO_DIR}/regenie_127phenotypes_lofteeHC_mac20_EUR_miss20per.parquet')
    .filter(pl.col('pval_fdr') <= 0.05)
    .select(['region', 'phenotype', 'pval_fdr'])
)

loftee_corrs = (
    pl.read_parquet(f'{LOCAL_ANNO_DIR}/regenie_127phenotypes_mac20_lofteeHC_EUR_correlations.parquet')
    .with_columns(
        loftee_corr_abs = pl.col('correlation').abs(),
        loftee_corr_dir = pl.col('correlation') / pl.col('correlation').abs(),
    )
    .select(['region', 'phenotype', 'loftee_corr_abs', 'loftee_corr_dir'])
)

gene_trait_df = (
    gene_trait_df
    .join(loftee_corrs, on=['region', 'phenotype'], how='inner')
    .drop_nans()
    .sort('loftee_corr_abs', descending=True)
    .unique(subset=["region"], keep="first", maintain_order=True)
    .select(['region', 'phenotype', 'loftee_corr_dir'])
)

# Create variant categories from merged annotations
# ClinVar Pathogenic (includes Pathogenic and Pathogenic/Likely_pathogenic)
clinvar_pathogenic = (
    anno
    .filter(pl.col('clinical_significance').str.starts_with('Pathogenic'))
    .select(['id', 'region'])
    .unique()
    .with_columns(category=pl.lit('ClinVar Pathogenic'))
)

# LOFTEE HC
loftee_hc_variants = (
    anno
    .filter(pl.col('loftee_hc') == 1)
    .select(['id', 'region'])
    .unique()
    .with_columns(category=pl.lit('LOFTEE HC'))
)

# ClinVar Benign (reference)
clinvar_benign = (
    anno
    .filter(pl.col('clinical_significance').str.starts_with('Benign'))
    .select(['id', 'region'])
    .unique()
    .with_columns(category=pl.lit('ClinVar Benign'))
)

variant_categories = pl.concat([clinvar_pathogenic, loftee_hc_variants, clinvar_benign])
print("Variant counts per category:")
print(variant_categories['category'].value_counts(sort=True))

# Load APPV and compute per-gene-trait mean direction-corrected z-scores
appv = pl.scan_parquet(f'{LOCAL_ANNO_DIR}/quant_pheno_INT_loftee_mac20_EURunrelated_miss20per_appv_small.parquet')

avg_pheno_df = (
    appv
    .filter(pl.col('n_individuals') <= mac)
    .select(['id', 'phenotype', 'mean_pheno_value'])
    .join(variant_categories.lazy(), on='id', how='inner')
    .join(gene_trait_df.lazy(), on=['region', 'phenotype'], how='inner')
    .with_columns(
        mean_pheno_value_dircor = pl.col('mean_pheno_value') * pl.col('loftee_corr_dir')
    )
    .group_by(['category', 'region', 'phenotype'])
    .agg(
        n_vars = pl.col('id').n_unique(),
        mean_pheno = pl.col('mean_pheno_value_dircor').mean(),
    )
    .collect(engine='streaming')
)

print("\nGene-trait associations per category:")
print(avg_pheno_df['category'].value_counts(sort=True))
avg_pheno_df

In [ ]:
# Order categories by median phenotype z-score
ordered_categories = (
    avg_pheno_df
    .group_by('category')
    .agg(pl.col('mean_pheno').median())
    .sort('mean_pheno', descending=False)
    .select('category')
    .to_series()
)

plot_df = avg_pheno_df.with_columns(
    pl.col('category').cast(pl.Enum(ordered_categories))
)

# Summary statistics
summary = (
    plot_df
    .group_by('category')
    .agg(
        n_gene_trait = pl.len(),
        median_zscore = pl.col('mean_pheno').median(),
        mean_zscore = pl.col('mean_pheno').mean(),
        ci_lower = pl.col('mean_pheno').quantile(0.025),
        ci_upper = pl.col('mean_pheno').quantile(0.975),
    )
    .sort('median_zscore', descending=True)
)
print(summary)

# Boxplot comparing ClinVar Pathogenic vs LOFTEE HC vs ClinVar Benign
(
    ggplot(
        plot_df,
        aes(x='category', y='mean_pheno')
    )
    + geom_boxplot(outlier_shape=None)
    + geom_hline(yintercept=0, linetype='dashed', color='red')
    + labs(
        x='',
        y='Mean phenotype z-score\n(direction corrected)',
        title='Average phenotype z-scores by variant category\n(per gene-trait association)'
    )
    + coord_flip()
    + theme_minimal()
    + theme(
        figure_size=(7, 4),
        axis_text=element_text(size=13),
        axis_title=element_text(size=13, lineheight=1.4),
        title=element_text(size=13, lineheight=1.4),
        plot_background=element_rect(fill="white", color="white"),
    )
)